# Pipeline de Préparation de Données - Prédiction Retards TGV

Ce notebook contient l'intégralité du flux de traitement des données :
1. **Nettoyage** des données brutes.
2. **Encodage** complet des variables catégorielles (OHE).
3. **Feature Engineering** temporel (Lags, Moyenne Mobile, Saisonnalité).

**Chemin du projet :** `C:/Users/Utilisateur/Documents/Albert School/B2/SEMESTRE 2/Achieving a ML Proof of Concept/ml-poc-project`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- CONFIGURATION DES CHEMINS ---
base_path = Path(r'C:/Users/Utilisateur/Documents/Albert School/B2/SEMESTRE 2/Achieving a ML Proof of Concept/ml-poc-project')
raw_data_path = base_path / 'data' / 'data_nettoyee.csv'
output_file = base_path / 'data' / 'processed_dataset.csv'

print(f'Chargement des données depuis : {raw_data_path}')

Chargement des données depuis : C:\Users\Utilisateur\Documents\Albert School\B2\SEMESTRE 2\Achieving a ML Proof of Concept\ml-poc-project\data\data_nettoyee.csv


## 1. Chargement et Nettoyage de Base
On gère les dates, on crée la cible (target) et on gère les valeurs manquantes.

In [2]:
df = pd.read_csv(raw_data_path, sep=',')
df['Date'] = pd.to_datetime(df['Date'])

# Suppression des colonnes vides
df = df.drop(columns=['Commentaire annulations', 'Commentaire retards au départ'], errors='ignore')

# Création de l'ID de ligne numérique
df['id_ligne'] = pd.factorize(df['Gare de départ'] + ' -> ' + df["Gare d'arrivée"])[0] + 1

# Calcul de la cible (Target : Taux de retard)
df['trains_ok'] = df['Nombre de circulations prévues'] - df['Nombre de trains annulés']
df['target'] = df["Nombre de trains en retard à l'arrivée"] / df['trains_ok']
df['target'] = df['target'].replace([np.inf, -np.inf], 0).fillna(0)

print(f'Données chargées : {len(df)} lignes.')

Données chargées : 8874 lignes.


## 2. Encodage des Variables Catégorielles
- **Commentaire** : Transformation en indicateur binaire (0 ou 1).
- **Service & Gares** : One-Hot Encoding complet.

In [3]:
# Traitement binaire du commentaire
df['commentaire'] = df["Commentaire retards à l'arrivée"].apply(
    lambda x: 1 if pd.notnull(x) and str(x).strip() not in ['', 'Aucun commentaire', 'nan'] else 0
)

# One-Hot Encoding (Service, Départ, Arrivée)
cols_to_ohe = ['Service', 'Gare de départ', "Gare d'arrivée"]
df = pd.get_dummies(df, columns=cols_to_ohe, prefix=['ser', 'dep', 'arr'])

print(f'Encodage terminé. Nombre de colonnes : {df.shape[1]}')

Encodage terminé. Nombre de colonnes : 147


## 3. Ingénierie Temporelle
Ajout des lags, moyennes mobiles et de la saisonnalité cyclique.

Dans une série temporelle, la valeur à l'instant $t$ dépend souvent des valeurs passées $t-1, t-2, ...$

- **Lags** : On décale la cible pour donner au modèle une vision du passé.
- **Moyenne Mobile** : On calcule la moyenne du retard sur les 3 derniers mois pour capturer la tendance.

Un mois est une variable circulaire. Pour que le modèle comprenne que Décembre (12) est proche de Janvier (1), on utilise :

$$x_{sin} = \sin\left(\frac{2\pi \cdot month}{12}\right)$$
$$x_{cos} = \cos\left(\frac{2\pi \cdot month}{12}\right)$$

In [4]:
# Tri indispensable pour les calculs temporels
df = df.sort_values(['id_ligne', 'Date'])

# Lags (1, 2, 3 mois)
for i in [1, 2, 3]:
    df[f'lag_{i}'] = df.groupby('id_ligne')['target'].shift(i)

# Moyenne mobile 3 mois
df['rolling_mean_3'] = df.groupby('id_ligne')['target'].transform(lambda x: x.shift(1).rolling(window=3).mean())

# Saisonnalité Sin/Cos
df['month'] = df['Date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)



print('Features temporelles ajoutées.')

Features temporelles ajoutées.


On ajoute des variables metiers : 
- **Volume de trafic** : Surcharge du réseau.
- **Annulations** : Signal de crises majeures.
- **Grands Départs** : Flag pour Juillet, Août, Décembre.

In [5]:
#Indicateurs métier

df['grand_depart'] = df['month'].isin([7, 8, 12]).astype(int)
df['taux_annulation'] = (df['Nombre de trains annulés'] / df['Nombre de circulations prévues']).fillna(0)
# 1. Surcharge Absolue : On utilise le volume brut de trains prévus
# Cela donne au modèle l'information de la densité de trafic globale.
df['surcharge_absolue'] = df['Nombre de circulations prévues']

# 2. Surcharge Relative : On compare le trafic du mois à la moyenne habituelle de cette ligne spécifique
# Un score > 1 indique un mois plus chargé que la normale (ex: pics de vacances).
# Cela permet de voir si l'infrastructure est plus sollicitée que d'habitude.
df['surcharge_relative'] = df.groupby('id_ligne')['Nombre de circulations prévues'].transform(lambda x: x / x.mean())

# 3. Ratio de saturation : trains prévus / durée moyenne du trajet
# Pour détecter si on essaie de faire passer trop de trains sur un trajet long/complexe.
df['densite_trafic'] = df['Nombre de circulations prévues'] / df['Durée moyenne du trajet']

print("Features de surcharge (absolue, relative et densité) ajoutées avec succès.")

Features de surcharge (absolue, relative et densité) ajoutées avec succès.


## 4. Nettoyage Final et Sauvegarde
Suppression des lignes sans historique et export vers `processed_dataset.csv`.

In [6]:
# Suppression des lignes avec NaNs (premiers mois de chaque trajet)
df_final = df.dropna(subset=['lag_3']).copy()

# Sauvegarde finale
df_final.to_csv(output_file, index=False)

print(f'✅ SUCCÈS : Fichier sauvegardé dans {output_file}')
df_final.head()

✅ SUCCÈS : Fichier sauvegardé dans C:\Users\Utilisateur\Documents\Albert School\B2\SEMESTRE 2\Achieving a ML Proof of Concept\ml-poc-project\data\processed_dataset.csv


,Date,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,Retard moyen de tous les trains au départ,Nombre de trains en retard à l'arrivée,Retard moyen des trains en retard à l'arrivée,Retard moyen de tous les trains à l'arrivée,...,lag_3,rolling_mean_3,month,month_sin,month_cos,grand_depart,taux_annulation,surcharge_absolue,surcharge_relative,densite_trafic
453,2018-04-01,141,841,279,200,9.498250,3.302906,109,25.859786,6.522420,...,0.169942,0.197080,4,8.660254e-01,-0.500000,0,0.331748,841,1.002284,5.964539
521,2018-05-01,140,844,252,107,15.510280,3.139949,99,22.769360,5.222072,...,0.261155,0.205082,5,5.000000e-01,-0.866025,0,0.298578,844,1.005860,6.028571
703,2018-06-01,142,844,156,126,18.932143,3.913978,129,27.692765,6.601163,...,0.160142,0.173774,6,1.224647e-16,-1.000000,0,0.184834,844,1.005860,5.943662
780,2018-07-01,146,817,160,337,16.589219,8.462963,185,37.885586,12.610299,...,0.193950,0.182893,7,-5.000000e-01,-0.866025,1,0.195838,817,0.973682,5.595890
910,2018-08-01,146,738,15,296,7.827140,3.119433,114,27.595322,5.986215,...,0.167230,0.212104,8,-8.660254e-01,-0.500000,1,0.020325,738,0.879531,5.054795
